In [1]:
import qlib
import pandas as pd
import numpy as np
from qlib.constant import REG_CN
from qlib.utils import exists_qlib_data, init_instance_by_config
from qlib.workflow import R
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord
from qlib.utils import flatten_dict
from qlib.contrib.model.gbdt import LGBModel
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.contrib.data.handler import Alpha158
from qlib.data import D

ModuleNotFoundError.  PyTorch models are skipped (optional: maybe installing pytorch can fix it).


In [2]:
qlib.init(provider_uri="~/.qlib/qlib_data/cn_data", region=REG_CN)

[80968:MainThread](2025-08-12 21:54:44,003) INFO - qlib.Initialization - [config.py:451] - default_conf: client.
[80968:MainThread](2025-08-12 21:54:44,015) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.
[80968:MainThread](2025-08-12 21:54:44,016) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/dynam1te/.qlib/qlib_data/cn_data')}


In [3]:
# 价格 / factor 对应与前复权的结果

df = D.features(instruments=['SH601033'], fields=['$open', '$close', "$factor"], start_time='2025-08-01', end_time='2025-08-06')

print(df)

                          $open    $close   $factor
instrument datetime                                
SH601033   2025-08-01  0.734431  0.739757  0.048413
           2025-08-04  0.736512  0.744259  0.048423
           2025-08-05  0.744608  0.746060  0.048414
           2025-08-06  0.747997  0.746060  0.048414


In [4]:
# 训练

market = "csi300"
benchmark = "SH000300"

model = LGBModel(
    loss="mse",
    colsample_bytree=0.8879,
    learning_rate=0.0421,
    subsample=0.8789,
    lambda_l1=205.6999,
    lambda_l2=580.9768,
    max_depth=8,
    num_leaves=210,
    num_threads=20,
    keep_training_booster=True
)

dataset = DatasetH( 
    handler=Alpha158(
        instruments="csi300",
        start_time="2018-01-01",
        end_time="2025-08-05",
        fit_start_time="2018-01-01",
        fit_end_time="2023-12-31",
        label = ["Ref($close, -3)/$close - 1"]
    ),
    segments={
	    "train": ("2018-01-01", "2023-12-31"),
	    "valid": ("2024-01-01", "2024-12-31"),
        "test": ("2025-01-01", "2025-08-06"),
    },
)

# start exp to train model
with R.start(experiment_name="train_model"):
    #R.log_params(**flatten_dict(task))
    model.fit(dataset)
    R.save_objects(trained_model=model)
    rid = R.get_recorder().id

[80968:MainThread](2025-08-12 21:59:23,472) INFO - qlib.timer - [log.py:127] - Time cost: 29.630s | Loading data Done
[80968:MainThread](2025-08-12 21:59:23,988) INFO - qlib.timer - [log.py:127] - Time cost: 0.113s | DropnaLabel Done
[80968:MainThread](2025-08-12 21:59:24,805) INFO - qlib.timer - [log.py:127] - Time cost: 0.816s | CSZScoreNorm Done
[80968:MainThread](2025-08-12 21:59:24,809) INFO - qlib.timer - [log.py:127] - Time cost: 1.336s | fit & process data Done
[80968:MainThread](2025-08-12 21:59:24,810) INFO - qlib.timer - [log.py:127] - Time cost: 30.968s | Init data Done
[80968:MainThread](2025-08-12 21:59:24,817) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name train_model.
[80968:MainThread](2025-08-12 21:59:24,819) INFO - qlib.workflow - [exp.py:258] - Experiment 546341030589329388 starts running ...
[80968:MainThread](2025-08-12 21:59:24,911) INFO - qlib.workflow - [recorder.py:345] - Recorder 36940bbe0d36405d80a537dd

Training until validation scores don't improve for 50 rounds
[20]	train's l2: 0.990594	valid's l2: 0.994738
[40]	train's l2: 0.986477	valid's l2: 0.993946
[60]	train's l2: 0.983108	valid's l2: 0.99358
[80]	train's l2: 0.979952	valid's l2: 0.993439
[100]	train's l2: 0.977185	valid's l2: 0.993486
[120]	train's l2: 0.974664	valid's l2: 0.993541
Early stopping, best iteration is:
[82]	train's l2: 0.979657	valid's l2: 0.993401


[80968:MainThread](2025-08-12 21:59:36,436) INFO - qlib.timer - [log.py:127] - Time cost: 0.228s | waiting `async_log` Done
